# GPT-Neo residual stream — every block, in memory

A decoder-only model, capturing every block at once: `n_layers x seq_len x
hidden` floats per passage. Token-shaped activations flatten padding away, so
the accumulated tensors only ever hold real tokens.

Downloads on first run: WikiText-2 (~5 MB) and GPT-Neo 125M weights (~500 MB).

In [ ]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel
from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples

MODEL = "EleutherAI/gpt-neo-125m"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token  # GPT-Neo ships without a pad token
model = AutoModelForCausalLM.from_pretrained(MODEL)  # has .logits, real next-token head


In [ ]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
hooked = HookedModel(model)

# The residual stream: every transformer block, plus the final layer norm.
num_layers = model.config.num_layers
LAYERS = [f"transformer.h.{i}" for i in range(num_layers)] + ["transformer.ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")

In [ ]:
# Cost before committing: one small batch gives the per-token shape.
pipeline = ActivationPipeline(model, LAYERS, output_type="token")
probe_loader = activation_loader(dataset, batch_size=1)
probe = pipeline.run([next(iter(probe_loader))], progress=False, cache=True).summary()
per_token = probe["bytes"].sum() / probe["bytes"].count()  # bytes already per-token here
print(probe)
print(f"{per_token / 1024:.1f} KB per real token")

In [ ]:
# cache=True accumulates every batch's activations in memory as the run
# progresses, flattened to one row per real token (padding dropped).
loader = activation_loader(dataset, batch_size=16)
activations = pipeline.run(loader, cache=True)
activations.summary()

In [ ]:
# Indexing the dataset slices out one passage's own tokens across every layer.
sample = activations[0]
print(
    "first passage ->",
    len(sample.activations),
    "layers,",
    tuple(next(iter(sample.activations.values())).shape),
    "each",
)

# Residual stream norm grows with depth - the usual GPT-2/GPT-Neo picture.
for name, tensor in sample.activations.items():
    print(f"  {name:<18} mean L2 norm = {tensor.norm(dim=-1).mean():6.2f}")